# Notebook 53: STRAT-004 Validation - Income Generation Strategy

**New Finding:** 1H @ 12% trail achieves:
- +3,129% total return (matches daily!)
- 14.6 trades/year (7x more than daily)
- 24 day average hold
- 50% win rate
- Sharpe 1.12

**Goal:** Validate this isn't overfitting and document STRAT-004 for income generation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

DATA_DIR = Path("../data")
HOURLY_DIR = DATA_DIR / "hourly"
DAILY_DIR = DATA_DIR / "daily"

## 1. Load Data

In [ ]:
# Load hourly
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    return df

# Load daily
def load_daily():
    price = pd.read_parquet(DAILY_DIR / "price.parquet")
    sopr = pd.read_parquet(DAILY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(DAILY_DIR / "sopr_sth.parquet")
    realized_loss = pd.read_parquet(DAILY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    return df

df_h1 = load_hourly()
df_d1 = load_daily()

print(f"Hourly: {len(df_h1):,} bars")
print(f"Daily: {len(df_d1):,} bars")

In [ ]:
# Add z-scores
def add_zscore(df, window):
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window, min_periods=window//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window, min_periods=window//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

df_h1 = add_zscore(df_h1, 365 * 24)  # 8760 hours = 1 year
df_d1 = add_zscore(df_d1, 365)       # 365 days = 1 year

# Filter to backtest period
START = "2019-01-01"
df_h1 = df_h1[df_h1.index >= START].dropna()
df_d1 = df_d1[df_d1.index >= START].dropna()

years = (df_d1.index.max() - df_d1.index.min()).days / 365.25
print(f"\nBacktest period: {years:.2f} years")

## 2. Define Both Strategies

In [ ]:
def get_metrics(pf, years):
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
    avg_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    
    winning = trades[trades["PnL"] > 0]
    losing = trades[trades["PnL"] < 0]
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_yr": len(trades) / years,
        "avg_days": avg_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(winning["PnL"].sum() / losing["PnL"].sum()) if len(losing) > 0 else np.inf,
        "avg_win": winning["Return"].mean() * 100 if len(winning) > 0 else 0,
        "avg_loss": losing["Return"].mean() * 100 if len(losing) > 0 else 0,
    }

def run_strategy(df, freq, trail):
    """Entry: SOPR < 1 AND STH-SOPR < 1 AND RL z-score > 0.5"""
    cond = (df["sopr"] < 1) & (df["sopr_sth"] < 1) & (df["rl_zscore"] > 0.5)
    entry = cond & ~cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None
    
    return vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail,
        sl_trail=True,
        freq=freq,
        init_cash=10000,
        fees=0.001
    )

In [ ]:
# Run both strategies
print("="*100)
print("HEAD-TO-HEAD: STRAT-002 vs STRAT-004")
print("="*100)

# STRAT-002: Daily 30% trail (Macro Swing)
pf_002 = run_strategy(df_d1, "1d", 0.30)
m_002 = get_metrics(pf_002, years)

# STRAT-004: Hourly 12% trail (Income)
pf_004 = run_strategy(df_h1, "1h", 0.12)
m_004 = get_metrics(pf_004, years)

print(f"\n{'Metric':<20} {'STRAT-002':>15} {'STRAT-004':>15} {'Winner':>15}")
print(f"{'(Daily 30%)':<20} {'':>15} {'(1H 12%)':<15}")
print("-"*70)

metrics_compare = [
    ("Total Return", "return", "%", True),
    ("CAGR", "cagr", "%", True),
    ("Sharpe Ratio", "sharpe", "", True),
    ("Max Drawdown", "max_dd", "%", False),
    ("Total Trades", "trades", "", True),
    ("Trades/Year", "trades_yr", "", True),
    ("Avg Hold (days)", "avg_days", "", False),
    ("Win Rate", "win_rate", "%", True),
    ("Profit Factor", "profit_factor", "", True),
]

for name, key, suffix, higher_better in metrics_compare:
    v_002 = m_002[key]
    v_004 = m_004[key]
    
    if higher_better:
        winner = "STRAT-004" if v_004 > v_002 else "STRAT-002" if v_002 > v_004 else "TIE"
    else:
        winner = "STRAT-004" if v_004 < v_002 else "STRAT-002" if v_002 < v_004 else "TIE"
    
    print(f"{name:<20} {v_002:>14.1f}{suffix} {v_004:>14.1f}{suffix} {winner:>15}")

## 3. Equity Curves

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Daily resample for fair comparison
pf_002.value().resample('D').last().plot(ax=ax, label=f"STRAT-002 Daily 30%: {m_002['return']:+,.0f}% ({m_002['trades']} trades)", linewidth=2)
pf_004.value().resample('D').last().plot(ax=ax, label=f"STRAT-004 1H 12%: {m_004['return']:+,.0f}% ({m_004['trades']} trades)", linewidth=2)

# Buy & Hold
bh = (df_d1["price"] / df_d1["price"].iloc[0]) * 10000
bh.plot(ax=ax, label=f"Buy & Hold: {(bh.iloc[-1]/10000-1)*100:+,.0f}%", linestyle='--', alpha=0.5)

ax.set_title("STRAT-002 vs STRAT-004: Equity Curves")
ax.set_ylabel("Portfolio Value ($)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 4. Trade Analysis

In [ ]:
print("\n" + "="*100)
print("STRAT-002 TRADES (Daily 30% Trail - Macro Capitulation)")
print("="*100)

trades_002 = pf_002.trades.records_readable
display_002 = trades_002[["Entry Timestamp", "Exit Timestamp", "Return", "PnL"]].copy()
display_002["Return %"] = display_002["Return"] * 100
display_002["Days"] = (trades_002["Exit Timestamp"] - trades_002["Entry Timestamp"]).dt.days
print(display_002[["Entry Timestamp", "Exit Timestamp", "Days", "Return %", "PnL"]].to_string())

In [ ]:
print("\n" + "="*100)
print("STRAT-004 TRADES (1H 12% Trail - Income Generation)")
print("="*100)

trades_004 = pf_004.trades.records_readable
display_004 = trades_004[["Entry Timestamp", "Exit Timestamp", "Return", "PnL"]].copy()
display_004["Return %"] = display_004["Return"] * 100
display_004["Days"] = (trades_004["Exit Timestamp"] - trades_004["Entry Timestamp"]).dt.days
print(display_004[["Entry Timestamp", "Exit Timestamp", "Days", "Return %", "PnL"]].to_string())

## 5. Robustness: Walk-Forward Validation

In [ ]:
# Test on different time periods
print("\n" + "="*100)
print("WALK-FORWARD VALIDATION: STRAT-004 (1H 12%)")
print("="*100)

periods = [
    ("2019-2020", "2019-01-01", "2021-01-01"),
    ("2020-2021", "2020-01-01", "2022-01-01"),
    ("2021-2022", "2021-01-01", "2023-01-01"),
    ("2022-2023", "2022-01-01", "2024-01-01"),
    ("2023-2024", "2023-01-01", "2025-01-01"),
    ("2024-2025", "2024-01-01", "2026-01-01"),
]

print(f"\n{'Period':<15} {'Return':>12} {'Trades':>8} {'Win%':>8} {'MaxDD':>10}")
print("-"*60)

for name, start, end in periods:
    df_period = df_h1[(df_h1.index >= start) & (df_h1.index < end)]
    if len(df_period) < 100:
        continue
    
    pf = run_strategy(df_period, "1h", 0.12)
    if pf:
        period_years = (df_period.index.max() - df_period.index.min()).days / 365.25
        m = get_metrics(pf, period_years)
        print(f"{name:<15} {m['return']:>+11.0f}% {m['trades']:>8} {m['win_rate']:>7.0f}% {m['max_dd']:>9.1f}%")
    else:
        print(f"{name:<15} {'No trades':>12}")

## 6. Income Projection

In [ ]:
print("\n" + "="*100)
print("INCOME PROJECTION")
print("="*100)

capital_levels = [50000, 100000, 250000, 500000, 1000000]

print(f"\n{'Strategy':<25} {'CAGR':>8}", end="")
for cap in capital_levels:
    print(f" {'$'+str(cap//1000)+'K':>12}", end="")
print()
print("-"*100)

for name, m in [("STRAT-002 (Daily 30%)", m_002), ("STRAT-004 (1H 12%)", m_004)]:
    print(f"{name:<25} {m['cagr']:>+7.1f}%", end="")
    for cap in capital_levels:
        annual = cap * m['cagr'] / 100
        print(f" ${annual:>11,.0f}", end="")
    print()

print(f"\n{'Per Trade Income @ $100K:':<25}")
print(f"  STRAT-002: ${100000 * (m_002['return']/100) / m_002['trades']:,.0f} per trade ({m_002['trades_yr']:.1f}/yr)")
print(f"  STRAT-004: ${100000 * (m_004['return']/100) / m_004['trades']:,.0f} per trade ({m_004['trades_yr']:.1f}/yr)")

## 7. Final Strategy Definitions

In [ ]:
print("\n" + "="*100)
print("DUAL STRATEGY FRAMEWORK")
print("="*100)

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                         STRAT-002: MACRO CAPITULATION                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Purpose:     Maximum returns, swing trading                                 ║
║  Timeframe:   Daily                                                          ║
║  Trail Stop:  30%                                                            ║
║                                                                              ║
║  Entry:       SOPR < 1 AND STH-SOPR < 1 AND Realized Loss z-score > 0.5     ║
║  Exit:        30% trailing stop                                              ║
║                                                                              ║
║  Expected:    ~2 trades/year, ~180 day holds, ~50% win rate                 ║
║  Best for:    Long-term wealth building, patient capital                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════════════════════════╗
║                         STRAT-004: INCOME GENERATION                         ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Purpose:     Active income, more frequent trades                            ║
║  Timeframe:   1 Hour                                                         ║
║  Trail Stop:  12%                                                            ║
║                                                                              ║
║  Entry:       SOPR < 1 AND STH-SOPR < 1 AND Realized Loss z-score > 0.5     ║
║  Exit:        12% trailing stop                                              ║
║                                                                              ║
║  Expected:    ~15 trades/year, ~24 day holds, ~50% win rate                 ║
║  Best for:    Active traders, income generation                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

PORTFOLIO ALLOCATION SUGGESTION:
  - Conservative: 100% STRAT-002
  - Balanced:     70% STRAT-002 + 30% STRAT-004
  - Active:       50% STRAT-002 + 50% STRAT-004
""")

In [ ]:
# Save strategy definitions
strategy_definitions = {
    "STRAT-002": {
        "name": "Macro Capitulation",
        "purpose": "Maximum returns, swing trading",
        "timeframe": "1d",
        "trail_stop": 0.30,
        "entry": {
            "sopr": "< 1",
            "sopr_sth": "< 1",
            "rl_zscore": "> 0.5"
        },
        "exit": "30% trailing stop",
        "expected": {
            "trades_per_year": m_002["trades_yr"],
            "avg_hold_days": m_002["avg_days"],
            "win_rate": m_002["win_rate"],
            "cagr": m_002["cagr"]
        }
    },
    "STRAT-004": {
        "name": "Income Generation",
        "purpose": "Active income, more frequent trades",
        "timeframe": "1h",
        "trail_stop": 0.12,
        "entry": {
            "sopr": "< 1",
            "sopr_sth": "< 1",
            "rl_zscore": "> 0.5"
        },
        "exit": "12% trailing stop",
        "expected": {
            "trades_per_year": m_004["trades_yr"],
            "avg_hold_days": m_004["avg_days"],
            "win_rate": m_004["win_rate"],
            "cagr": m_004["cagr"]
        }
    }
}

import json
with open("../data/strategy_definitions.json", "w") as f:
    json.dump(strategy_definitions, f, indent=2)

print("Strategy definitions saved to data/strategy_definitions.json")